In [1]:
import h5py
import traceback
import numpy as np
import plotly.graph_objects as go
import matplotlib as mpl
mpl.use('Qt5Agg')  # Use Qt5 backend for GUI to work properly
import matplotlib.pyplot as plt
import scipy.signal as sc
from scipy.optimize import curve_fit
from scipy.ndimage import filters 
import tifffile as tiff
from scipy.signal import find_peaks
import plotly.express as px
import os
import csv
import pandas as pd
from roipoly import MultiRoi
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy.interpolate import interp1d
import ast
from AnalasysFunction import plot_spike_shapes_plotly_4states,BurstC,clean_and_parse,motorSp,plotFR,devideTr,plotVolCal,VolToCalIdx,CalSmooth,CorrWindow,ChooseSpk,CalInt,VolToCalIdx,CalAmp,calculate_firing_rate,ChooseCom,LongLIST,SingleSpk,linear_model,quadratic_model,exponential_model,MeanRes,lagOptimaizre
from TRY import LongLIST
from NewinternueronsAnalsys import analyze_block
from scipy.optimize import curve_fit
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import pearsonr, linregress,ttest_ind
import pickle
from matplotlib.widgets import Slider, Button

from scipy.ndimage import median_filter
from qixin_spike_detection_4 import sst_spike_correction_gui,detect_bursts_from_vm,complex_bursts_detection, refine_single_spikes,spike_height_calculation2, detect_complex_spikes, refine_all_spikes


In [2]:
def select_params_interactive_extended(trace, spike_idx, frame_rate, 
                                       init_CS=0.5, init_SS=0.3, 
                                       init_threshold=0.5, init_cb_amp_threshold=0.3,
                                       init_cb_duration_threshold=20,init_threshold_CS=6,init_threshold_SS=6,
                                       inint_ss_cap=0.5,CB_detection_method='simple',
                                       SS_detection_method='simple',CS_SP_TH=0.5, save_html_path=None,n=''):
    """
    Opens an interactive window to tune spike detection parameters.
    Press ENTER when done, or window auto-closes after 30 seconds.
    
    NOTE: ISI threshold is FIXED at 20 ms (10 samples at 500 Hz).
    """
    plt.close('all')
    
    if len(spike_idx) == 0:
        return init_CS, init_SS, init_threshold, init_cb_amp_threshold, init_cb_duration_threshold
    
    selected_params = {
        'pnorm_CS': init_CS,
        'pnorm_SS': init_SS,
        'ST_cs': init_threshold_CS,
        'ST_ss': init_threshold_SS,
        
        'ss_cap': inint_ss_cap,
        'threshold': float(init_threshold),
        'cb_amp_threshold': init_cb_amp_threshold,
        'cb_duration_threshold': init_cb_duration_threshold,
        'isi_threshold_ms': 20  # HARDCODED
    }
    final_spikes = {
    "all_spikes_vm": np.array([], dtype=int),
    "complex_spikes_vm": np.array([], dtype=int),
    "simple_spikes_vm": np.array([], dtype=int),
    "complex_burst_vm":np.array([], dtype=int),
    "burst_metrics" : [],
    "vm"    : np.array([], dtype=float),
    "heights": np.array([], dtype=float),
    "SNR": np.array([], dtype=float),
        }

    
    # Create figure with room for sliders
    fig, ax = plt.subplots(figsize=(15, 10))
    plt.subplots_adjust(bottom=0.50)
    
    t_axis = np.arange(len(trace)) / frame_rate
    trace_line, = ax.plot(t_axis, trace, 'k', lw=0.5, alpha=0.6, label='Trace (norm)')
    
    # Scatter plots for spikes
    # Input spike candidates (for sanity-checking index alignment)
    scat_in = ax.scatter([], [], c='0.3', s=12, label='Input spike_idx', zorder=3)
    scat_ss = ax.scatter([], [], c='b', s=30, label='Single Spikes (Qixin)', zorder=5)
    scat_cs = ax.scatter([], [], c='r', s=50, marker='x', label='Complex Spikes (Qixin)', zorder=6)
    scat_vm_simple = ax.scatter([], [], c='cyan', s=20, marker='o', label='Simple Spikes (Vm)', zorder=4)
    scat_vm_complex = ax.scatter([], [], c='magenta', s=40, marker='x', label='Complex Spikes (Vm)', zorder=7)
    ss_cap_line = ax.axhline(np.nan, color='b', ls='--', lw=1, alpha=0.6, label='SS cap (raw)')
    
    count_text = ax.text(0.02, 0.95, "Initializing...", transform=ax.transAxes,
                         bbox=dict(facecolor='white', alpha=0.8), fontsize=10)
    burst_spans = []
    latest_state = {}

    def save_latest_state_to_html():
        if not save_html_path:
            return
        if not isinstance(latest_state, dict) or 'trace_norm' not in latest_state:
            print('No latest detection state available to save.')
            return
        try:
            out_dir = os.path.dirname(save_html_path)
            if out_dir:
                os.makedirs(out_dir, exist_ok=True)

            trace_norm = np.asarray(latest_state['trace_norm'], dtype=float).ravel()
            t = np.arange(len(trace_norm)) / frame_rate

            fig_html = go.Figure()
            fig_html.add_trace(go.Scattergl(x=t, y=trace_norm, mode='lines', name='Trace (norm)', line=dict(color='black', width=1), opacity=0.6))

            def _add_points(name, idx, color, symbol, size):
                if idx is None:
                    return
                idx = np.asarray(idx, dtype=int)
                idx = idx[(idx >= 0) & (idx < len(trace_norm))]
                if idx.size == 0:
                    return
                fig_html.add_trace(go.Scattergl(x=t[idx], y=trace_norm[idx], mode='markers', name=name, marker=dict(color=color, symbol=symbol, size=size)))

            _add_points('Input spike_idx', latest_state.get('idx_in'), 'rgba(80,80,80,0.8)', 'circle', 6)
            _add_points('Single Spikes (Qixin)', latest_state.get('idx_ss'), 'blue', 'circle', 8)
            _add_points('Complex Spikes (Qixin)', latest_state.get('idx_cs'), 'red', 'x', 10)
            _add_points('Simple Spikes (Vm)', latest_state.get('idx_vm_simple'), 'cyan', 'circle-open', 8)
            _add_points('Complex Spikes (Vm)', latest_state.get('idx_vm_complex'), 'magenta', 'x', 10)

            starts = latest_state.get('burst_starts', None)
            ends = latest_state.get('burst_ends', None)
            if starts is not None and ends is not None:
                starts = np.asarray(starts, dtype=float).ravel()
                ends = np.asarray(ends, dtype=float).ravel()
                for s, e in zip(starts, ends):
                    try:
                        fig_html.add_vrect(x0=float(s)/frame_rate, x1=float(e)/frame_rate, fillcolor='yellow', opacity=0.2, line_width=0, layer='below')
                    except Exception:
                        fig_html.add_shape(type='rect', x0=float(s)/frame_rate, x1=float(e)/frame_rate, y0=0, y1=1, yref='paper', fillcolor='yellow', opacity=0.2, line_width=0, layer='below')

            p_ss_cap = latest_state.get('p_ss_cap', None)
            if p_ss_cap is not None and np.isfinite(p_ss_cap):
                try:
                    fig_html.add_hline(y=float(p_ss_cap), line_dash='dash', line_color='blue', opacity=0.5)
                except Exception:
                    fig_html.add_shape(type='line', x0=float(t[0]), x1=float(t[-1]), y0=float(p_ss_cap), y1=float(p_ss_cap), line=dict(color='blue', dash='dash', width=1), opacity=0.5)

            latest_candidates = []
            for label, key in [('Complex Spikes (Vm)', 'idx_vm_complex'), ('Complex Spikes (Qixin)', 'idx_cs'), ('Simple Spikes (Vm)', 'idx_vm_simple'), ('Single Spikes (Qixin)', 'idx_ss')]:
                arr = latest_state.get(key, None)
                if arr is None:
                    continue
                arr = np.asarray(arr, dtype=int)
                arr = arr[(arr >= 0) & (arr < len(trace_norm))]
                if arr.size:
                    latest_candidates.append((int(arr.max()), label))
            if latest_candidates:
                latest_idx, latest_label = max(latest_candidates, key=lambda x: x[0])
                fig_html.add_trace(go.Scattergl(x=[t[latest_idx]], y=[trace_norm[latest_idx]], mode='markers+text', name='Latest spike', marker=dict(color='gold', size=14, symbol='star'), text=[f'Latest: {latest_label}'], textposition='top center'))

            params = latest_state.get('params', {}) if isinstance(latest_state.get('params', {}), dict) else {}
            title = 'Latest spike detected & classified (interactive)' + f" | CS={params.get('pnorm_CS', np.nan):.2f} SS={params.get('pnorm_SS', np.nan):.2f} T={params.get('threshold', np.nan):.2f}"
            fig_html.update_layout(title=title, xaxis_title='Time (s)', yaxis_title='Trace (norm)', template='plotly_white', height=650, legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0))
            fig_html.write_html(save_html_path, include_plotlyjs=True, full_html=True)
            print(f'Saved HTML figure: {save_html_path}')
        except Exception as e:
            print(f'Failed to save HTML figure: {e}')


    ax.set_title(f"Input Spikes: {len(spike_idx)} | Adjust Sliders & Press ENTER (or wait 30s)")
    ax.legend(loc='upper right', fontsize=8)
    
    # Create sliders for CS/SS detection
    ax_cs = plt.axes([0.2, 0.45, 0.6, 0.03])
    ax_ss = plt.axes([0.2, 0.40, 0.6, 0.03])
    ax_th = plt.axes([0.2, 0.35, 0.6, 0.03])
    ax_cb_amp = plt.axes([0.2, 0.30, 0.6, 0.03])
    ax_cb_dur = plt.axes([0.2, 0.25, 0.6, 0.03])
    ax_cb_st = plt.axes([0.2, 0.2, 0.6, 0.03])
    ax_ss_st = plt.axes([0.2, 0.15, 0.6, 0.03])
    ax_cb_cap = plt.axes([0.2, 0.1, 0.6, 0.03])

    slider_cs = Slider(ax_cs, 'P-Norm CS', 0.0, 4, valinit=init_CS)
    slider_ss = Slider(ax_ss, 'P-Norm SS', 0.0, 2, valinit=init_SS)
    slider_th = Slider(ax_th, 'CS Height Thresh', 0.0, 4.5, valinit=init_threshold)
    slider_cb_amp = Slider(ax_cb_amp, 'CB Amp Thresh', 0.0, 4.5, valinit=init_cb_amp_threshold, valstep=0.05)
    slider_cb_dur = Slider(ax_cb_dur, 'CB Duration (ms)', 5, 300, valinit=init_cb_duration_threshold, valstep=5)
    slider_cs_st = Slider(ax_cb_st, 'simple threshol CS', 0.0, 10, valinit=init_threshold_CS)
    slider_ss_st = Slider(ax_ss_st, 'simple threshol SS', 0.0, 10, valinit= init_threshold_SS)
    slider_ss_cap = Slider(ax_cb_cap, 'SS min height (norm)', 0.0, 4, valinit= inint_ss_cap, valstep=0.05)
    # Add text label for ISI threshold (not adjustable)
    fig.text(0.2, 0.24, 'ISI Threshold (FIXED): 20 ms', fontsize=9, weight='bold', 
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    def run_detection(p_cs, p_ss, p_th, p_cb_amp, p_cb_dur, p_ss_cap, p_st_ss, p_st_cs,CB_detection_method,
                      SS_detection_method):
        """Run full spike detection pipeline with Vm burst detection"""
        try:
            # Always apply simple thresholds during tuning
            CB_detection_method = "simple"
            SS_detection_method = "simple"

            # Step 1: Complex burst detection
            c_bursts, _ = complex_bursts_detection(
                trace, spike_idx, frame_rate,
                pnorm=p_cs, process_window=30, plotflag=False,CB_detection_method=CB_detection_method,
                simple_threshold=p_st_cs
            )
            
            # Step 2: Refine single spikes
            r_ss, t_noCS = refine_single_spikes(
                trace, spike_idx, c_bursts, frame_rate,
                process_window=30, pnorm=p_ss,f_hp=20, min_spikes=5, plotflag=False
                ,SS_detection_method=SS_detection_method,simple_threshold_SS=p_st_ss,SS_height_cap=None
            )
            
            # Step 3: Calculate spike heights
            trace_mf = c_bursts.get('trace_mf', median_filter(trace, size=11))
            heights, snr = spike_height_calculation2(
                r_ss, trace, trace_mf, t_noCS, frame_rate
            )
            #height_scale = float(np.nanmedian(heights)) if np.size(heights) else 1.0
            #height_scale = max(height_scale, 1e-12)
            height_scale = float(np.nanmedian(heights)) if np.size(heights) else 1.0
            height_scale = max(height_scale, 1e-12)
            trace_norm = trace / height_scale
            # Apply SS cap on normalized trace (0..1 slider makes sense here)
            if p_ss_cap is not None:
                idx_ss = r_ss.astype(int)
                idx_ss = idx_ss[(idx_ss >= 0) & (idx_ss < len(trace_norm))]
                r_ss = idx_ss[trace_norm[idx_ss] >= float(p_ss_cap)]
            ss_cap_raw = float(p_ss_cap) * height_scale
            
            # Step 4: Detect complex spikes
            cs_spikes_raw = detect_complex_spikes(
                trace_norm, c_bursts, np.ones_like(trace_norm), threshold=p_th, plotflag=False
            )
            
            # Step 5: Refine all spikes
            c_bursts_final, r_ss_final, all_cs, all_s = refine_all_spikes(
                c_bursts, cs_spikes_raw, r_ss
            )
            
            # Step 6: Detect bursts from Vm (with ISI = 20 ms hardcoded)
            simple_spikes_vm, complex_spikes_vm, all_spikes_vm, trace_snr, vm, burst_metrics, complex_bursts_vm = detect_bursts_from_vm(
                trace_norm,
                np.ones_like(trace_norm),
                c_bursts_final,
                all_s,
                frame_rate,
                highpass=1.0,
                median_window=11,
                cb_amp_threshold=p_cb_amp,
                cb_duration_threshold=p_cb_dur,
                isi_threshold_ms=20,
                baseline_subtract=True,
                baseline_window_ms=20,
                baseline_percentile=10,
                merge_SS_ms = 20,
                merge_CB_ms = 5,
            )
            
            return c_bursts_final, r_ss_final, all_cs, complex_bursts_vm, simple_spikes_vm, complex_spikes_vm, ss_cap_raw, trace_norm,all_spikes_vm,burst_metrics,vm,heights,trace_snr
        
        except Exception as e:
            print(f"Error in run_detection: {e}")
            import traceback
            traceback.print_exc()
            trace_norm = np.asarray(trace, dtype=float)
            return {}, np.array([], dtype=int), np.array([], dtype=int), {}, np.array([], dtype=int), np.array([], dtype=int), float("nan"), trace_norm, np.array([], dtype=int), [], np.array([], dtype=float), np.array([], dtype=float), np.array([], dtype=float)

    def update(val):
        p_cs = slider_cs.val
        p_ss = slider_ss.val
        p_th = float(slider_th.val)
        p_cb_amp = slider_cb_amp.val
        p_cb_dur = slider_cb_dur.val
        p_ss_cap = slider_ss_cap.val
        p_st_ss = slider_ss_st.val
        p_st_cs = slider_cs_st.val

                
        
        selected_params.update({
            'pnorm_CS': p_cs,
            'pnorm_SS': p_ss,
            'threshold': p_th,
            'cb_amp_threshold': p_cb_amp,
            'cb_duration_threshold': p_cb_dur,
            'ST_cs': p_st_cs,
            'ST_ss': p_st_ss,
            'ss_cap': p_ss_cap
        })
        
        try:
            c_bursts, r_ss, all_cs, c_bursts_vm, simple_spikes_vm, complex_spikes_vm, ss_cap_raw, trace_norm, all_spikes_vm,burst_metrics,vm,heights,SNR = run_detection(
                p_cs, p_ss, p_th, p_cb_amp, p_cb_dur, p_ss_cap, p_st_ss, p_st_cs,CB_detection_method,
                SS_detection_method
            )
            final_spikes["all_spikes_vm"] = np.asarray(all_spikes_vm, dtype=int).copy()
            final_spikes["complex_spikes_vm"] = np.asarray(complex_spikes_vm, dtype=int).copy()
            final_spikes["simple_spikes_vm"] = np.asarray(simple_spikes_vm, dtype=int).copy()
            final_spikes["complex_burst_vm"] = c_bursts_vm
            final_spikes["vm"] = np.asarray(vm, dtype=float).copy()
            final_spikes["burst_metrics"] = burst_metrics.copy()
            final_spikes["heights"] = np.asarray(heights, dtype=float).copy()
            final_spikes["SNR"] = np.asarray(SNR, dtype=float).copy()

            trace_norm = np.asarray(trace_norm, dtype=float).ravel()
            idx_in = np.asarray(spike_idx, dtype=int)
            idx_in = idx_in[(idx_in >= 0) & (idx_in < len(trace_norm))]
            if idx_in.size > 0:
                scat_in.set_offsets(np.c_[t_axis[idx_in], trace_norm[idx_in]])
                scat_in.set_visible(True)
            else:
                scat_in.set_visible(False)
            
            idx_ss = np.array([], dtype=int)
            idx_cs = np.array([], dtype=int)
            idx_vm_simple = np.array([], dtype=int)
            idx_vm_complex = np.array([], dtype=int)
            starts = np.array([], dtype=int)
            ends = np.array([], dtype=int)

            # --- QIXIN SPIKE COUNTS ---
            n_ss = len(r_ss) if isinstance(r_ss, np.ndarray) else 0
            n_cs = len(all_cs) if isinstance(all_cs, np.ndarray) else 0
            
            # --- VM BURST SPIKE COUNTS ---
            n_vm_simple = len(simple_spikes_vm) if isinstance(simple_spikes_vm, np.ndarray) else 0
            n_vm_complex = len(complex_spikes_vm) if isinstance(complex_spikes_vm, np.ndarray) else 0
            n_bursts_vm = len(c_bursts_vm.get('starts', [])) if isinstance(c_bursts_vm, dict) else 0
            
            count_text.set_text(
                f"QIXIN: SS={n_ss} | CS={n_cs}\n"
                f"VM-BURST: Simple={n_vm_simple} | Complex={n_vm_complex} | Bursts={n_bursts_vm}\n"
                f"CS={p_cs:.2f} | SS={p_ss:.2f} | T={p_th:.2f}\n"
                f"CB_Amp={p_cb_amp:.2f} | CB_Dur={p_cb_dur:.0f}ms | ISI=20ms (fixed)\n"
                + (f"SS_cap(norm)={p_ss_cap:.2f} ~ raw {ss_cap_raw:.3g}" if ss_cap_raw is not None else f"SS_cap(norm)={p_ss_cap:.2f}")
            )

            # Update displayed trace to normalized trace
            trace_line.set_ydata(trace_norm)
            ax.relim()
            ax.autoscale_view(scalex=False, scaley=True)

            # Update SS cap threshold line (normalized units)
            if ss_cap_raw is not None and np.isfinite(ss_cap_raw):
                ss_cap_line.set_ydata([float(p_ss_cap), float(p_ss_cap)])
                ss_cap_line.set_visible(True)
            else:
                ss_cap_line.set_visible(False)

            # --- PLOT QIXIN SINGLE SPIKES (BLUE) ---
            if n_ss > 0:
                idx_ss = r_ss.astype(int)
                idx_ss = idx_ss[(idx_ss >= 0) & (idx_ss < len(trace_norm))]
                if len(idx_ss) > 0:
                    scat_ss.set_offsets(np.c_[t_axis[idx_ss], trace_norm[idx_ss]])
                    scat_ss.set_visible(True)
                else:
                    scat_ss.set_visible(False)
            else:
                scat_ss.set_visible(False)

            # --- PLOT QIXIN COMPLEX SPIKES (RED X) ---
            if n_cs > 0:
                idx_cs = all_cs.astype(int)
                idx_cs = idx_cs[(idx_cs >= 0) & (idx_cs < len(trace_norm))]
                if len(idx_cs) > 0:
                    scat_cs.set_offsets(np.c_[t_axis[idx_cs], trace_norm[idx_cs]])
                    scat_cs.set_visible(True)
                else:
                    scat_cs.set_visible(False)
            else:
                scat_cs.set_visible(False)
            
            # --- PLOT VM SIMPLE SPIKES (CYAN CIRCLES) ---
            if n_vm_simple > 0:
                idx_vm_simple = simple_spikes_vm.astype(int)
                idx_vm_simple = idx_vm_simple[(idx_vm_simple >= 0) & (idx_vm_simple < len(trace_norm))]
                if len(idx_vm_simple) > 0:
                    scat_vm_simple.set_offsets(np.c_[t_axis[idx_vm_simple], trace_norm[idx_vm_simple]])
                    scat_vm_simple.set_visible(True)
                else:
                    scat_vm_simple.set_visible(False)
            else:
                scat_vm_simple.set_visible(False)
            
            # --- PLOT VM COMPLEX SPIKES (MAGENTA X) ---
            if n_vm_complex > 0:
                idx_vm_complex = complex_spikes_vm.astype(int)
                idx_vm_complex = idx_vm_complex[(idx_vm_complex >= 0) & (idx_vm_complex < len(trace_norm))]
                if len(idx_vm_complex) > 0:
                    scat_vm_complex.set_offsets(np.c_[t_axis[idx_vm_complex], trace_norm[idx_vm_complex]])
                    scat_vm_complex.set_visible(True)
                else:
                    scat_vm_complex.set_visible(False)
            else:
                scat_vm_complex.set_visible(False)
            
            # --- PLOT VM BURST SPANS (YELLOW BANDS) ---
            for span in burst_spans:
                span.remove()
            burst_spans.clear()
            
            if isinstance(c_bursts_vm, dict):
                starts = c_bursts_vm.get('starts', np.array([]))
                ends = c_bursts_vm.get('ends', np.array([]))
                for s, e in zip(starts, ends):
                    span = ax.axvspan(s/frame_rate, e/frame_rate, color='yellow', alpha=0.3)
                    burst_spans.append(span)
            
            latest_state.clear()
            latest_state.update({
                'trace_norm': trace_norm,
                'idx_in': idx_in,
                'idx_ss': idx_ss,
                'idx_cs': idx_cs,
                'idx_vm_simple': idx_vm_simple,
                'idx_vm_complex': idx_vm_complex,
                'burst_starts': starts,
                'burst_ends': ends,
                'p_ss_cap': float(p_ss_cap) if p_ss_cap is not None else None,
                'params': selected_params.copy(),
            })

            fig.canvas.draw_idle()
        except Exception as e:
            count_text.set_text(f"ERROR: {str(e)[:80]}")
            fig.canvas.draw_idle()

    slider_cs.on_changed(update)
    slider_ss.on_changed(update)
    slider_th.on_changed(update)
    slider_cb_amp.on_changed(update)
    slider_cb_dur.on_changed(update)
    slider_ss_cap.on_changed(update)
    slider_ss_st.on_changed(update)
    slider_cs_st.on_changed(update) 
    
    update(None)
    
    def on_key(event):
        """Capture ENTER key press"""
        if event.key == 'enter':
            plt.close(fig)

    fig.canvas.mpl_connect('key_press_event', on_key)
    
    # --- SAFE WAIT LOOP (non-blocking with timeout) ---
    plt.show(block=False)
    
    for _ in range(40000):  # 30 seconds max
        if not plt.fignum_exists(fig.number):
            break
        plt.pause(0.015)
    else:
        print("  ⚠ Interactive timeout – using current parameter values")
    
    plt.close('all')
    save_latest_state_to_html()

    return (
        selected_params['pnorm_CS'],
        selected_params['pnorm_SS'],
        selected_params['threshold'],
        selected_params['cb_amp_threshold'],
        selected_params['cb_duration_threshold'],
        selected_params['ss_cap'],
        selected_params['ST_ss'],
        selected_params['ST_cs'],
        final_spikes["all_spikes_vm"],
        final_spikes["complex_spikes_vm"],
        final_spikes["simple_spikes_vm"],
        final_spikes["complex_burst_vm"],
        final_spikes["vm"],
        final_spikes["burst_metrics"],
        final_spikes["heights"],
        final_spikes["SNR"]    
    )


In [3]:
def get_Vol_Cal_SS(ss_list, Vol_trace, Cal_trace, volAx, calAx, preW=100, postW=500, save_path=None):
    """
    Extract spike-aligned voltage and calcium traces.
    Handles edge spikes with variable window lengths.
    Returns actual lengths for proper time-axis alignment per spike.
    """
    calTl = []
    volTl = []
    calIdx = []
    actual_lens = []  # Track actual length of each trace
    
    for i, l in enumerate(ss_list):
        l = int(l)
        
        # Clamp to valid range (allows variable window)
        volStart = max(0, l - preW)
        volEnd = min(len(Vol_trace) - 1, l + postW)
        
        # Calculate actual windows used (may be shorter at edges)
        actual_pre = l - volStart  # Actual pre-spike window
        actual_post = volEnd - l   # Actual post-spike window
        actual_total = volEnd - volStart + 1
        
        # Convert to calcium indices
        calS = VolToCalIdx(volStart, volAx, calAx)
        calE = VolToCalIdx(volEnd, volAx, calAx)
        calE = min(calE, len(Cal_trace) - 1)
        
        # Extract traces WITHOUT padding
        vol_segment = Vol_trace[volStart:volEnd + 1]
        cal_segment = Cal_trace[calS:calE + 1]
        
        volTl.append(vol_segment)
        calTl.append(cal_segment)
        calIdx.append(VolToCalIdx(l, volAx, calAx))
        actual_lens.append({
            'idx': i,
            'vol_len': len(vol_segment),
            'cal_len': len(cal_segment),
            'pre': actual_pre,
            'post': actual_post
        })
    
    if save_path is not None:
        with open(save_path, "w") as f:
            f.write(f"preW = {preW}\n")
            f.write(f"postW = {postW}\n")
            f.write(f"actual_lengths = {actual_lens}\n")
    
    return calTl, volTl, calIdx, actual_lens  # ← Return actual lengths!

def get_Vol_Cal_Burst(Burst_list,Vol_trae,Cal_trace,volAx,calAx, preW = 100, postW = 500,save_path = None):
    calTl = []
    volTl = []
    for i,l in enumerate(Burst_list):
        volStart = l[0] - preW
        volend = l[-1]+ postW
        calS = VolToCalIdx(volStart,volAx,calAx)
        calE = VolToCalIdx(volend,volAx,calAx)
        volTl.append(Vol_trae[volStart:volend])
        calTl.append(Cal_trace[calS:calE])
        if save_path is not None:
            with open(save_path, "w") as f:
                f.write(f"preW = {preW}\n")
                f.write(f"postW = {postW}\n")
    return calTl,volTl
def flatten_traces(traces):
    """Flattens nested lists into a simple list of 1D numpy arrays."""
    out = []
    for x in traces:
        if x is None:
            continue
        if isinstance(x, (list, tuple)):
            out.extend(flatten_traces(x))
        else:
            arr = np.asarray(x).ravel()
            if arr.size > 0:
                out.append(arr.astype(float))
    return out

def auc_window(v, t_ms, win, baseline_win=(-50,-20), positive_only=True):
    v = np.asarray(v, float)
    t = np.asarray(t_ms, float)

    bi = np.where((t >= baseline_win[0]) & (t <= baseline_win[1]))[0]
    wi = np.where((t >= win[0]) & (t <= win[1]))[0]
    if bi.size < 3 or wi.size < 3:
        return np.nan

    baseline = np.nanmedian(v[bi])
    y = v - baseline
    y = y[wi]
    if positive_only:
        y = np.maximum(y, 0)
    return np.trapz(y, t[wi])
def remove_spike_linear_interp(v, t_ms, peak_search_win=(-5, 5), cut_win=(-2, 6)):
    v = np.asarray(v, dtype=float).copy()
    t_ms = np.asarray(t_ms, dtype=float)

    # --- tolerate mismatch by truncating to the shorter length ---
    L = min(len(v), len(t_ms))
    v = v[:L]
    t = t_ms[:L]

    # ---- find spike peak near 0 ms ----
    ps_idx = np.where((t >= peak_search_win[0]) & (t <= peak_search_win[1]))[0]
    if ps_idx.size == 0:
        pk = int(np.nanargmax(v))
    else:
        pk = ps_idx[int(np.nanargmax(v[ps_idx]))]

    t_pk = t[pk]

    # ---- cut region around peak ----
    cut_start = t_pk + cut_win[0]
    cut_end   = t_pk + cut_win[1]
    cut_idx = np.where((t >= cut_start) & (t <= cut_end))[0]
    if cut_idx.size == 0:
        return v

    i0, i1 = int(cut_idx[0]), int(cut_idx[-1])

    # ---- edge safety ----
    if i0 <= 0 or i1 >= (L - 1):
        v[i0:i1+1] = np.nan
        return v

    # ---- linear interpolation ----
    y0 = v[i0 - 1]
    y1 = v[i1 + 1]
    v[i0:i1+1] = np.linspace(y0, y1, i1 - i0 + 1)

    return v
def classify_subthreshold_with_spike_removed_samewin(
    V_traces, t_ms,
    signal_win=(-20, 100),
    noise_win=(-140, -20),
    cut_win=(-2, 4),
    k=3.0,
    positive_only=True):
    """
    Classify spikes based on AUC, handling variable-length traces.
    """
    V_list = flatten_traces(V_traces)
    
    if len(V_list) == 0:
        return np.array([], dtype=bool), np.array([]), np.array([]), np.nan
    
    # Get common length for all traces
    common_len = max(len(v) for v in V_list)
    t_ms = np.asarray(t_ms, dtype=float)
    
    # If t_ms is shorter than traces, pad it
    if len(t_ms) < common_len:
        dt = t_ms[1] - t_ms[0] if len(t_ms) > 1 else 1.0
        t_ms_extended = np.arange(common_len) * dt + t_ms[0]
        t_ms = t_ms_extended
    
    # Truncate to common length and align
    auc_sig = []
    auc_nse = []
    
    for v in V_list:
        v_padded = np.asarray(v, dtype=float)
        
        # Pad if shorter than common length
        if len(v_padded) < common_len:
            v_padded = np.pad(v_padded, (0, common_len - len(v_padded)), 
                             mode='constant', constant_values=np.nan)
        else:
            v_padded = v_padded[:common_len]
        
        # Clean spike
        v_clean = remove_spike_linear_interp(v_padded, t_ms[:common_len], cut_win=cut_win)
        
        # Calculate AUC
        sig_auc = auc_window(v_clean, t_ms[:common_len], signal_win, 
                            baseline_win=(-50, -20), positive_only=positive_only)
        nse_auc = auc_window(v_clean, t_ms[:common_len], noise_win, 
                            baseline_win=(-50, -20), positive_only=positive_only)
        
        auc_sig.append(sig_auc)
        auc_nse.append(nse_auc)
    
    auc_sig = np.array(auc_sig, dtype=float)
    auc_nse = np.array(auc_nse, dtype=float)
    
    # Threshold from noise distribution
    mu = np.nanmean(auc_nse)
    sd = np.nanstd(auc_nse)
    thr = mu + k * sd
    
    mask = auc_sig > thr
    
    return mask, auc_sig, auc_nse, thr

In [4]:
def load_and_refine_pyr_spikes(
    path,trace_vol,trace_cal,
    frame_rate=500,
    default_pn_CS=0.5,
    default_pn_SS=0.3,
    default_thresh=0.5,
    default_cb_amp_threshold=0.3,
    default_cb_duration_threshold=20,
    CB_detection_method = 'simple', # 'simple' or 'volpy_based'
    SS_detection_method = 'simple',
    simple_threshold_cs = 6,
    simple_threshold_SS = 6,
    ss_height_cap = 0.5,
    f_hp = 20,
    preW=75,
    postW=100,
    burst_window_ms=15,
    save_dir=None,
    interactive=True,
    spikeID = None,params_pr = None,
    save_only=False,
    final_spikes=None,
    selected_params=None,
    name =''

):
    """
    Complete pipeline for pyramidal cell spike refinement using qixin_spike_detection:
    1. Load voltage & calcium traces
    2. Load or detect initial spikes
    3. Run interactive spike detection (if interactive=True)
    4. Refine spikes using qixin functions
    5. Detect bursts from Vm (FINAL spike detection)
    6. Save refined spikes as PKL
    7. Extract spike-aligned voltage & calcium traces
    8. Classify spikes (single vs burst vs complex)
    9. Choose sparse single spikes (no ADP)
    
    Parameters
    ----------
    path : str
        Path to cell directory (contains calTraceDF.csv, volTraceDF.csv, etc.)
    frame_rate : int
        Sampling rate in Hz (default 500 for voltage)
    default_pn_CS : float
        Initial complex spike p-norm parameter
    default_pn_SS : float
        Initial single spike p-norm parameter
    default_thresh : float
        Initial CS detection threshold
    default_cb_amp_threshold : float
        Initial complex burst amplitude threshold (default 0.3)
    default_cb_duration_threshold : float
        Initial complex burst duration threshold (default 20 ms)
    preW, postW : int
        Pre/post window samples for spike-aligned extraction
    burst_window_ms : int
        Window for burst detection (ms)
    save_dir : str or None
        Directory to save outputs (defaults to path)
    interactive : bool
        If True, open interactive tuning window; else use defaults
    
    Returns
    -------
    results_dict : dict with comprehensive spike analysis results using Vm burst detection
    """
    
    if save_dir is None:
        save_dir = path
    os.makedirs(save_dir, exist_ok=True)

    # ===== SAVE-ONLY: WRITE PICKLE WITHOUT RE-RUNNING =====
    # Does NOT load anything from disk. Keeps the same saved pickle structure.
    # Provide `final_spikes` (outputs of select_params_interactive_extended OR selected from your loaded cell__CS_detection.pkl).
    if save_only:
        if final_spikes is None or not isinstance(final_spikes, dict):
            raise ValueError('save_only=True requires final_spikes dict')

        p = selected_params if isinstance(selected_params, dict) else (params_pr if isinstance(params_pr, dict) else {})
        p_CS = p.get('pnorm_CS', default_pn_CS)
        p_SS = p.get('pnorm_SS', default_pn_SS)
        p_Th = p.get('threshold', default_thresh)
        p_cb_amp = p.get('cb_amp_threshold', default_cb_amp_threshold)
        p_cb_dur = p.get('cb_duration_threshold', default_cb_duration_threshold)
        p_ss_cap = p.get('ss_cap', ss_height_cap)
        p_ST_ss = p.get('simple_threshold_ss', simple_threshold_SS)
        p_ST_cs = p.get('simple_threshold_cs', simple_threshold_cs)

        # Accept either vm_* keys (new) or non-vm keys (older/select_params output).
        all_spikes = final_spikes.get('vm_all_spikes', final_spikes.get('all_spikes', []))
        complex_spikes = final_spikes.get('vm_complex_spikes', final_spikes.get('complex_spikes', []))
        simple_spikes = final_spikes.get('vm_simple_spikes', final_spikes.get('simple_spikes', []))
        complex_bursts = final_spikes.get('vm_burst_dict', final_spikes.get('complex_bursts', final_spikes.get('complex_bursts_dict', {})))
        burst_metrics = final_spikes.get('burst_metrics', final_spikes.get('vm_burst_metrics', []))
        heights = final_spikes.get('heights', final_spikes.get('spike_heights', final_spikes.get('spike_heights_interpolated', [])))
        snr = final_spikes.get('snr', final_spikes.get('SNR', final_spikes.get('spike_SNR', final_spikes.get('SNR_interpolated', []))))
        input_spikes = final_spikes.get('input_spikes', spikeID if spikeID is not None else [])

        save_data = {
            'trace_vol': trace_vol,
            'trace_cal': trace_cal,
            'input_spikes': np.asarray(input_spikes, dtype=int),
            'vm_burst_dict': complex_bursts if isinstance(complex_bursts, dict) else {},
            'vm_simple_spikes': np.asarray(simple_spikes, dtype=int),
            'vm_complex_spikes': np.asarray(complex_spikes, dtype=int),
            'vm_all_spikes': np.asarray(all_spikes, dtype=int),
            'vm_burst_metrics': burst_metrics if burst_metrics is not None else [],
            'spike_heights': np.asarray(heights, dtype=float) if heights is not None else np.array([], dtype=float),
            'spike_SNR': np.asarray(snr, dtype=float) if snr is not None else np.array([], dtype=float),
            'detection_params': {
                'pnorm_CS': p_CS,
                'pnorm_SS': p_SS,
                'threshold': p_Th,
                'cb_amp_threshold': p_cb_amp,
                'cb_duration_threshold': p_cb_dur,
                'ss_cap': p_ss_cap,
                'simple_threshold_ss': p_ST_ss,
                'simple_threshold_cs': p_ST_cs,
                'isi_threshold_ms': 20,
                'frame_rate': frame_rate,
                'preW': preW,
                'postW': postW,
                'detection_method': p.get('detection_method', 'Vm_burst_detection')
            }
        }

        pkl_path = os.path.join(save_dir, f'spike_detection_refined_new{name}.pkl')
        with open(pkl_path, 'wb') as f:
            pickle.dump(save_data, f, protocol=pickle.HIGHEST_PROTOCOL)
        return save_data


    # ===== 2. LOAD OR DETECT INITIAL SPIKES =====
    spike_pkl_path = os.path.join(path, 'cell__CS_detection.pkl')
    initial_spikes = spikeID
    prev_params = params_pr
    
    if initial_spikes is None or len(initial_spikes) == 0:
        threshold = np.percentile(np.abs(np.diff(trace_vol)), 95)
        initial_spikes = np.where(np.abs(np.diff(trace_vol)) > threshold)[0].tolist()
    
    initial_spikes = np.asarray(initial_spikes, dtype=int)
    init_CS = prev_params['pnorm_CS'] if prev_params else default_pn_CS
    init_SS = prev_params['pnorm_SS'] if prev_params else default_pn_SS
    init_thresh = prev_params['threshold'] if prev_params else default_thresh
    init_cb_amp = prev_params.get('cb_amp_threshold', default_cb_amp_threshold) if prev_params else default_cb_amp_threshold
    init_cb_dur = prev_params.get('cb_duration_threshold', default_cb_duration_threshold) if prev_params else default_cb_duration_threshold
    init_simple_thresh_CS = prev_params.get('simple_threshold_cs', simple_threshold_cs) if prev_params else simple_threshold_cs
    init_simple_thresh_SS =  prev_params.get('simple_threshold_ss', simple_threshold_SS)if prev_params else simple_threshold_SS
    init_ss_cap = prev_params.get('ss_cap', ss_height_cap)if prev_params else ss_height_cap
   

    #2.5 FIRS ROUND RUN
    complex_bursts_dict, segment_bounds = complex_bursts_detection(
                            trace_vol, initial_spikes, frame_rate,
                            pnorm=init_CS,
                            process_window=30,
                            plotflag=False,
                            CB_detection_method=CB_detection_method,
                            simple_threshold=init_simple_thresh_CS
                        )

    refined_SS, trace_noCS = refine_single_spikes(trace_vol, initial_spikes, complex_bursts_dict, frame_rate, process_window=30, pnorm=init_CS, f_hp=f_hp, min_spikes=5, plotflag=False,
                                                   SS_detection_method=SS_detection_method, simple_threshold_SS=init_simple_thresh_SS )
    initial_spikes= refined_SS
    spike_heights_interpolated, SNR_interpolated = spike_height_calculation2(refined_SS, trace_vol, complex_bursts_dict['trace_mf'], trace_noCS, frame_rate, plotflag=False )

    # ===== 3. INTERACTIVE REFINEMENT =====
    if interactive and len(initial_spikes) > 0:
        init_CS = prev_params['pnorm_CS'] if prev_params else default_pn_CS
        init_SS = prev_params['pnorm_SS'] if prev_params else default_pn_SS
        init_thresh = prev_params['threshold'] if prev_params else default_thresh
        init_cb_amp = prev_params.get('cb_amp_threshold', default_cb_amp_threshold) if prev_params else default_cb_amp_threshold
        init_cb_dur = prev_params.get('cb_duration_threshold', default_cb_duration_threshold) if prev_params else default_cb_duration_threshold
        
        p_CS, p_SS, p_Th, p_cb_amp, p_cb_dur,p_ss_cap,p_ST_ss,p_ST_cs,all_spikes,complex_spikes,simple_spikes,complex_bursts,vm,burst_metrics,heights,snr = select_params_interactive_extended(
            trace= trace_vol,spike_idx= initial_spikes, frame_rate=frame_rate,
            init_CS=init_CS,
            init_SS=init_SS,
            init_threshold=init_thresh,
            init_cb_amp_threshold=init_cb_amp,
            init_cb_duration_threshold=init_cb_dur,
            init_threshold_CS=init_simple_thresh_CS,
            init_threshold_SS=init_simple_thresh_SS,
            inint_ss_cap=init_ss_cap,
            CB_detection_method=CB_detection_method,
            SS_detection_method=SS_detection_method, 
            save_html_path=os.path.join(save_dir, f'latest_spike_detected_and_classified{name}.html')
        )
    # else:
    #     p_CS = prev_params['pnorm_CS'] if prev_params else default_pn_CS
    #     p_SS = prev_params['pnorm_SS'] if prev_params else default_pn_SS
    #     p_Th = prev_params['threshold'] if prev_params else default_thresh
    #     p_cb_amp = prev_params.get('cb_amp_threshold', default_cb_amp_threshold) if prev_params else default_cb_amp_threshold
    #     p_cb_dur = prev_params.get('cb_duration_threshold', default_cb_duration_threshold) if prev_params else default_cb_duration_threshold
    
    # # # ===== 4. REFINE SPIKES USING QIXIN FUNCTIONS =====
    # # try:
    # #     # Step 1: Detect complex bursts
    # #     complex_bursts_dict, _ = complex_bursts_detection(
    # #         trace_vol, initial_spikes, frame_rate,CB_detection_method=CB_detection_method,  
    # #         simple_threshold=p_ST_cs,
    # #         pnorm=p_CS, process_window=30, plotflag=False
    # #     )
        
    # #     # Step 2: Refine single spikes
    # #     refined_SS, trace_noCS = refine_single_spikes(
    # #         trace_vol, initial_spikes, complex_bursts_dict, frame_rate,
    # #         process_window=30, pnorm=p_SS, min_spikes=5,SS_detection_method=SS_detection_method,
    # #         simple_threshold=p_ST_ss,SS_height_cap=p_ss_cap, plotflag=False
    # #     )
        
    # #     # Step 3: Calculate spike heights
    # #     heights, snr = spike_height_calculation2(
    # #         refined_SS, trace_vol, complex_bursts_dict.get('trace_mf'), 
    # #         trace_noCS, frame_rate
    # #     )
    # #     trace_vol = trace_vol/heights
    # #     # Step 4: Detect complex spikes
    # #     CS_spikes = detect_complex_spikes(
    # #         trace_vol, complex_bursts_dict, np.ones_like(trace_vol),
    # #         threshold=p_Th, plotflag=False
    # #     )
        
    # #     # Step 5: Refine all spikes
    # #     complex_bursts_dict_final, refined_SS_final, all_CS_spikes, all_spikes = refine_all_spikes(
    # #         complex_bursts_dict, CS_spikes, refined_SS
    # #     )
        
    # # except Exception as e:
    # #     print(f"✗ Error during Qixin refinement: {e}")
    # #     refined_SS_final = initial_spikes
    # #     all_CS_spikes = np.array([], dtype=int)
    # #     all_spikes = initial_spikes
    # #     complex_bursts_dict_final = {}
    # #     heights = None
    # #     snr = None
    
    # # # ===== 5. FINAL: DETECT BURSTS FROM VM (PRIMARY SPIKE DETECTION) =====
    # # try:
    # #     simple_spikes_vm, complex_spikes_vm, all_spikes_vm, trace_snr, vm, burst_metrics, complex_bursts_vm = detect_bursts_from_vm(
    # #         trace_vol,
    # #         np.ones_like(trace_vol),
    # #         complex_bursts_dict_final,
    # #         all_spikes,
    # #         frame_rate,
    # #         highpass=1.0,
    # #         median_window=11,
    # #         cb_amp_threshold=p_cb_amp,
    # #         cb_duration_threshold=p_cb_dur,
    # #         isi_threshold_ms=20,  # FIXED
    # #         baseline_subtract=True,
    # #         baseline_window_ms=20,
    # #         baseline_percentile=10,
    # #         merge_SS_ms = 20,
    # #         merge_CB_ms = 5,
    # #     )
        
    #     # ✅ USE VM BURST DETECTION AS FINAL SPIKES
    #     final_all_spikes =  all_spikes_vm
    #     final_simple_spikes = simple_spikes_vm
    #     final_complex_spikes = complex_spikes_vm
    #     final_burst_dict = final_complex_bursts
        
    # except Exception as e:
    #     print(f"✗ Error during Vm burst detection: {e}")
    #     # Fallback to Qixin detection
    #     final_all_spikes = all_spikes
    #     final_simple_spikes = refined_SS_final
    #     final_complex_spikes = all_CS_spikes
    #     final_burst_dict = complex_bursts_dict_final
    #     burst_metrics = []
    
    # ===== 6. SAVE REFINED SPIKE DETECTION PKL =====
    save_data = {
        'trace_vol': trace_vol,
        'trace_cal': trace_cal,
        'input_spikes': initial_spikes,
        
        'vm_burst_dict': complex_bursts,  # ← PRIMARY
        'vm_simple_spikes': simple_spikes,  # ← PRIMARY
        'vm_complex_spikes': complex_spikes,  # ← PRIMARY
        'vm_all_spikes': all_spikes,  # ← PRIMARY (used for analysis)
        'vm_burst_metrics': burst_metrics,
        'spike_heights': heights,
        'spike_SNR': snr,
        'detection_params': {
            'pnorm_CS': p_CS,
            'pnorm_SS': p_SS,
            'threshold': p_Th,
            'cb_amp_threshold': p_cb_amp,
            'cb_duration_threshold': p_cb_dur,
            'ss_cap': p_ss_cap,
            'simple_threshold_ss': p_ST_ss,
            'simple_threshold_cs': p_ST_cs,
            'isi_threshold_ms': 20,  # FIXED
            'frame_rate': frame_rate,
            'preW': preW,
            'postW': postW,
            'detection_method': 'Vm_burst_detection'  # ← Document final method
        }
    }
    
    pkl_path = os.path.join(save_dir, f'spike_detection_refined_new{name}.pkl')
    with open(pkl_path, 'wb') as f:
        pickle.dump(save_data, f, protocol=pickle.HIGHEST_PROTOCOL)
    
    # ===== 7. CLASSIFY BURST SPIKES (using final Vm spikes) =====
    burst_window_samples = int(burst_window_ms * frame_rate / 1000)
    MyBurstSpike, numberOFspike = BurstC(all_spikes.tolist(), burst_window_samples)
    
    # Extract single vs burst indices (from VM detection)
    single_spike_idx = np.array([all_spikes[i] for i, n in enumerate(numberOFspike) if n == 1], dtype=int)
    burst_spike_idx = np.array([all_spikes[i] for i, n in enumerate(numberOFspike) if n > 1], dtype=int)
    
    # ===== 8. EXTRACT SPIKE-ALIGNED TRACES =====
    vol_ax = np.linspace(0, len(trace_vol)/frame_rate, len(trace_vol))
    cal_ax = np.linspace(0, len(trace_cal)/30, len(trace_cal))

    cal_vol_SS, cal_vol_V, cal_indices, actual_lens = get_Vol_Cal_SS(
        single_spike_idx, trace_vol, trace_cal,
        vol_ax, cal_ax,
        preW=preW, postW=postW
    )

    # ===== 9. CLASSIFY SPIKES (SPARSE vs ADP) - HANDLE VARIABLE LENGTHS =====
    if len(cal_vol_V) > 0:
        max_len = max(len(v) for v in cal_vol_V)
        first_pre_ms = actual_lens[0]['pre'] * 1000 / frame_rate
        t_ms_base = np.arange(max_len) * 1000 / frame_rate - first_pre_ms
        
        sparse_mask = []
        auc_sig_all = []
        auc_nse_all = []
        auc_thr = np.nan
        
        #print(f"  Processing {len(cal_vol_V)} traces with variable lengths...")
        
        for spike_idx_i, v_trace in enumerate(cal_vol_V):
            if len(v_trace) == 0:
                sparse_mask.append(False)
                auc_sig_all.append(np.nan)
                auc_nse_all.append(np.nan)
                continue
            
            actual_pre_ms = actual_lens[spike_idx_i]['pre'] * 1000 / frame_rate
            t_ms_spike = np.arange(len(v_trace)) * 1000 / frame_rate - actual_pre_ms
            
            try:
                mask_i, auc_sig_i, auc_nse_i, thr_i = classify_subthreshold_with_spike_removed_samewin(
                    [v_trace], t_ms_spike,
                    signal_win=(-20, 100),
                    noise_win=(-140, -20),
                    cut_win=(-2, 4),
                    k=3.0,
                    positive_only=True
                )
                
                sparse_mask.append(bool(mask_i[0]))
                auc_sig_all.append(auc_sig_i[0])
                auc_nse_all.append(auc_nse_i[0])
                auc_thr = thr_i
                
            except Exception as e:
                print(f"    ⚠ Warning spike {spike_idx_i}: {e}")
                sparse_mask.append(False)
                auc_sig_all.append(np.nan)
                auc_nse_all.append(np.nan)
        
        sparse_mask = np.array(sparse_mask, dtype=bool)
        auc_sig = np.array(auc_sig_all, dtype=float)
        auc_noise = np.array(auc_nse_all, dtype=float)
        
        valid_noise = auc_noise[~np.isnan(auc_noise)]
        if len(valid_noise) > 0:
            mu = np.nanmean(valid_noise)
            sd = np.nanstd(valid_noise)
            auc_thr = mu + 3.0 * sd
        
        sparse_single_spikes = single_spike_idx[sparse_mask] if len(sparse_mask) > 0 else np.array([])
        adp_single_spikes = single_spike_idx[~sparse_mask] if len(sparse_mask) > 0 else np.array([])
    else:
        sparse_mask = np.array([], dtype=bool)
        auc_sig = np.array([])
        auc_noise = np.array([])
        auc_thr = np.nan
        sparse_single_spikes = np.array([])
        adp_single_spikes = np.array([])

    pct_sparse = 100 * sparse_mask.sum() / len(sparse_mask) if len(sparse_mask) > 0 else 0
    
    # ===== 10. COMPILE RESULTS (Using VM Burst Detection) =====
    results_dict = {
        'trace_vol': trace_vol,
        'trace_cal': trace_cal,
        'spike_data': save_data,
        'refined_spikes': all_spikes,  # ✅ VM BURST DETECTION (FINAL)
        'single_spikes': simple_spikes,  # ✅ VM BURST DETECTION (FINAL)
        'complex_spikes': complex_spikes,  # ✅ VM BURST DETECTION (FINAL)
        'burst_spikes': burst_spike_idx,
        'sparse_single_spikes': sparse_single_spikes,
        'adp_single_spikes': adp_single_spikes,
        'vol_traces': cal_vol_V,
        'cal_traces': cal_vol_SS,
        'cal_indices': cal_indices,
        'auc_sig': auc_sig,
        'auc_noise': auc_noise,
        'auc_threshold': auc_thr,
        'sparse_mask': sparse_mask,
        'burst_metrics': burst_metrics,
        'params': {
            'frame_rate': frame_rate,
            'preW': preW,
            'postW': postW,
            'burst_window_ms': burst_window_ms,
            'pnorm_CS': p_CS,
            'pnorm_SS': p_SS,
            'threshold': p_Th,
            
            'cb_amp_threshold': p_cb_amp,
            'cb_duration_threshold': p_cb_dur,
            'isi_threshold_ms': 20,
            'detection_method': 'Vm_burst_detection'  # ← FINAL METHOD
        }
    }
    
    # Save results summary
    summary_path = os.path.join(save_dir, f'spike_classification_summary_new{name}.pkl')
    with open(summary_path, 'wb') as f:
        pickle.dump(results_dict, f, protocol=pickle.HIGHEST_PROTOCOL)
    
    # print(f"\n✓ SPIKE DETECTION COMPLETE (Vm Burst Method)")
    # print(f"  Total spikes: {len(final_all_spikes)}")
    # print(f"  Simple spikes: {len(final_simple_spikes)}")
    # print(f"  Complex spikes: {len(final_complex_spikes)}")
    # print(f"  Sparse (no ADP): {len(sparse_single_spikes)} ({pct_sparse:.1f}%)")
    
    return results_dict

In [5]:
def Split_cal(chane_p,VolT,CalT,volax,calax,mot,spikeIdx):
    calMot = []
    volMot = []
    volRes = []
    calRes = []
    spikeMot = []
    spikeRes = []
    if chane_p[0]<5:
        chane_p = chane_p[1:]
    for i in range(len(chane_p)+1):
        if i == 0:
            sIDX = 0
            eIdx = chane_p[i]
        elif i > 0 and i < len(chane_p):
            sIDX = chane_p[i-1]
            eIdx = chane_p[i]
        elif i ==len(chane_p):
            sIDX = chane_p[-1]
            eIdx = len(VolT)-1
        #print(s)
        calS = VolToCalIdx(sIDX,volax,calax)
        calE = VolToCalIdx(eIdx,volax,calax)
        if mot[sIDX+7] == 1:
            #print(f'motor{sIDX}')
            spikeId = [i for i in spikeIdx if i >= sIDX and i < eIdx]
            spikeMot.append(np.array(spikeId) - sIDX)
            calMot.append(CalT[calS:calE+1])
            volMot.append(VolT[sIDX:eIdx+1])
        if mot[sIDX+7] == 0:
            #print(f'rest{sIDX}')
            
            spikeId = [i for i in spikeIdx if i >= sIDX and i < eIdx]
            spikeRes.append(np.array(spikeId) - sIDX)
            calRes.append(CalT[calS:calE+1])
            #print(len(calRes))
            volRes.append(VolT[sIDX:eIdx+1])
            #print(len(volRes))
    return calMot,calRes,spikeMot,spikeRes,volMot,volRes


In [6]:
DB = pd.read_csv(r'Z:\Adam-Lab-Shared\Data\Michal_Rubin\Dendrites\PyrB.csv')
values = DB['SNR'].tolist()
r = DB
awakePyr = r['Notes']
bsPyr = list(r['brainState'])
pathPyr = list(r['Link'])
MotorSSVol = []
MotorSSCal = []
RestSSVol = []
RestSSCal = []
AwakeSSVol = []
AwakeSSCal = []
AnstSSVol = []
AnstSSCal = []
Rest_result = []
Motor_result = []
Anst_result = []
Awake_result = []
AllCellAllData= []


In [ ]:
for i in range(49,len(pathPyr)):
    l = pathPyr[i]
    #print(l)
    TracePathCal = os.path.join(l,'calTraceDF.csv')
    TracePathVol = os.path.join(l,'volTraceDF.csv')
    TracePathCalM = os.path.join(l,'calMask.csv')
    TracePathVolM = os.path.join(l,'volMask.csv')
    parentP = os.path.dirname(l)
    MotPath = os.path.join(parentP,'Sync','MotorId.csv')
    trace_vol_full = np.array(pd.read_csv(TracePathVol)).flatten().astype(float)
    trace_cal_full = np.array(pd.read_csv(TracePathCal)).flatten().astype(float)
    vol_mask = np.array(pd.read_csv(TracePathVolM)).flatten().astype(bool)
    cal_mask = np.array(pd.read_csv(TracePathCalM)).flatten().astype(bool)
    # Apply masks
    trace_vol = trace_vol_full[vol_mask]
    trace_cal = trace_cal_full[cal_mask]
    motor = pd.read_csv(MotPath, header=None).iloc[:, 0]
    motor = motor[0:np.size(trace_vol,0)]
    VolAX = np.linspace(0, (len(trace_vol)/500), len(trace_vol)) 
    CalAX = np.linspace(0, (len(trace_cal)/30), len(trace_cal))
    # path =os.path.join(l,r'cell__CS_detection.pkl')
    # # 2. Open the file
    # with open(path, 'rb') as f:
    #     data = pickle.load(f)
    
    # # 3. Retrieve the specific key
    # # This is currently a list of lists (e.g., [ [1, 5, 10], [2, 8, 20] ])
    # raw_spikes_structure = data['all_spikes']
    # long_all_spikes = []
    # if isinstance(raw_spikes_structure, list):
    #     for sublist in raw_spikes_structure:
    #         if isinstance(sublist, (list, np.ndarray)):
    #             long_all_spikes.extend(sublist)
    #         else:
    #             long_all_spikes.append(sublist)
    # else:
    #     long_all_spikes = list(raw_spikes_structure)
    # params_list = data['CS_detection_params']['selected_params_per_cell']
    # if len(params_list) > 0:
    #     prev_params = params_list[-1]   # last used params
    #     init_CS = prev_params['pnorm_CS']
    #     init_SS = prev_params['pnorm_SS']
    #     init_thresh = prev_params['threshold']
    # # Sort to ensure chronological order
    # long_all_spikes = sorted(list(set(long_all_spikes)))
    # long_all_spikes = sorted(long_all_spikes)
    if bsPyr[i].lower()=='motor':
        TracePathCal = os.path.join(l,'changepoint.csv')
        changeP = pd.read_csv(TracePathCal)
        changeP = np.array(changeP)
        changeP = np.array(changeP).flatten().tolist()
        calMot,calRes, spikeMot,spikeRes,vmot,volR = Split_cal(changeP,trace_vol,trace_cal,VolAX,CalAX,motor,long_all_spikes)
        
        RestFRpath =os.path.join(l,r'FiringRateRest.csv')
        MotorFRpath =os.path.join(l,r'FiringRateMotor.csv')
        dfMot = pd.read_csv(MotorFRpath)
        MotorFR = dfMot['fr'].tolist()
        dfRest = pd.read_csv(RestFRpath)
        RestFR = dfRest['fr'].tolist()
        g = 0
        r = 0
        for k in range(len(MotorFR)):
            
            TracePathVol = os.path.join(l,f'volTraceMot{k}.csv')
            TracePathCal = os.path.join(l,f'calTraceMot{k}.csv')
            trace_vol_full = np.array(pd.read_csv(TracePathVol)).flatten().astype(float)
            trace_cal_full = np.array(pd.read_csv(TracePathCal)).flatten().astype(float)
            VolAX = np.linspace(0, (len(trace_vol_full)/500), len(trace_vol_full)) 
            CalAX = np.linspace(0, (len(trace_cal_full)/30), len(trace_cal_full))
            N = f'm{k}'
            
            curr_path_pkl = os.path.join(l, f'spike_detection_refined_new{N}.pkl')
            if os.path.exists(curr_path_pkl):
                with open(curr_path_pkl, 'rb') as f:
                    data = pickle.load(f)
                spike_indices = data['vm_all_spikes']
                
                params_list = data['detection_params']

                if len(params_list) > 0:
                    prev_params = params_list
                    init_CS = prev_params['pnorm_CS']
                    init_SS = prev_params['pnorm_SS']
                    init_thresh = prev_params['threshold']
                    INIT_CB_AMP = prev_params['cb_amp_threshold']
                    INIT_CB_DUR = prev_params['cb_duration_threshold']
                    init_thresh_SS = prev_params['simple_threshold_ss']
                    init_thresh_CS = prev_params['simple_threshold_cs']
                    init_thresh_ISI = prev_params['isi_threshold_ms']
            

                # Sort to ensure chronological order
                currSp = sorted(list(set(spike_indices)))
                currSp = sorted(spike_indices)
            else:
                currSp = spikeMot[k].tolist()
            if len(trace_vol_full) < 13 * 500:
                #print(f"SKIP {l} r{k}: VolTrace too short ({len(trace_vol_full)/500:.2f} s, {len(trace_vol_full)} frames)")
                continue
            plt.close('all')
            results = load_and_refine_pyr_spikes(
                    path=l,trace_vol=trace_vol_full,trace_cal=trace_cal_full,
                    frame_rate=500,
                    interactive=True,  # Set to True for interactive tuning
                    preW=75,
                    postW=100,spikeID=currSp,params_pr=prev_params,name=N

                )
            Motor_result.append(results)
            MotorSSVol.extend(results['vol_traces'])
            MotorSSCal.extend(results['cal_traces'])
            AllCellAllData.append(results)
        for j in range(len(RestFR)):
            TracePathVol = os.path.join(l,f'volTraceRest{j}.csv')
            TracePathCal = os.path.join(l,f'calTraceRest{j}.csv')
            trace_vol_full = np.array(pd.read_csv(TracePathVol)).flatten().astype(float)
            trace_cal_full = np.array(pd.read_csv(TracePathCal)).flatten().astype(float)
            VolAX = np.linspace(0, (len(trace_vol_full)/500), len(trace_vol_full)) 
            CalAX = np.linspace(0, (len(trace_cal_full)/30), len(trace_cal_full))
            N = f'r{j}'
            curr_path_pkl = os.path.join(l, f'spike_detection_refined_new{N}.pkl')
            if os.path.exists(curr_path_pkl):
                with open(curr_path_pkl, 'rb') as f:
                    data = pickle.load(f)
                spike_indices = data['vm_all_spikes']
                
                params_list = data['detection_params']
                if len(params_list) > 0:
                    prev_params = params_list
                    init_CS = prev_params['pnorm_CS']
                    init_SS = prev_params['pnorm_SS']
                    init_thresh = prev_params['threshold']

                # Sort to ensure chronological order
                currSp = sorted(list(set(spike_indices)))
                currSp = sorted(spike_indices)
            else:
                currSp = spikeRes[j].tolist()
            if len(trace_vol_full) < 10 * 500:
                print(f"SKIP {l} r{j}: VolTrace too short ({len(trace_vol_full)/500:.2f} s, {len(trace_vol_full)} frames)")
                continue
            plt.close('all')
            results = load_and_refine_pyr_spikes(
                    path=l,trace_vol=trace_vol_full,trace_cal=trace_cal_full,
                    frame_rate=500,
                    interactive=True,  # Set to True for interactive tuning
                    preW=75,
                    postW=100,spikeID=currSp,params_pr=prev_params,name=N

                )
            Rest_result.append(results)
            RestSSVol.extend(results['vol_traces'])
            RestSSCal.extend(results['cal_traces'])
            AllCellAllData.append(results)
    elif bsPyr[i].lower()=='awake':
        N=''
        plt.close('all')
        spikePath = os.path.join(l, 'spike_detection_refined_new.pkl')
        spikePath2 = os.path.join(l, 'spike_detection_refined_newr1.pkl')
        if os.path.exists(spikePath) or os.path.exists(spikePath2):
            if os.path.exists(spikePath):
                with open(spikePath, 'rb') as f:
                    spike_data = pickle.load(f)
                long_all_spikes = spike_data['vm_all_spikes']
                params_list = spike_data['detection_params']
                if len(params_list) > 0:
                    prev_params = params_list
                    init_CS = prev_params['pnorm_CS']
                    init_SS = prev_params['pnorm_SS']
                    init_thresh = prev_params['threshold']
            else:
                with open(spikePath2, 'rb') as f:
                    spike_data = pickle.load(f)
                long_all_spikes = spike_data['vm_all_spikes']
                params_list = spike_data['detection_params']
                if len(params_list) > 0:
                    prev_params = params_list
                    init_CS = prev_params['pnorm_CS']
                    init_SS = prev_params['pnorm_SS']
                    init_thresh = prev_params['threshold']
        results = load_and_refine_pyr_spikes(
                    path=l,trace_vol=trace_vol,trace_cal=trace_cal,
                    frame_rate=500,
                    interactive=True,  # Set to True for interactive tuning
                    preW=75,
                    postW=100,spikeID=long_all_spikes,params_pr=prev_params,name=N

                )
        Awake_result.append(results)
        AwakeSSVol.extend(results['vol_traces'])   
        AwakeSSCal.extend(results['cal_traces'])
        AllCellAllData.append(results)
    elif bsPyr[i].lower()=='anst':
        plt.close('all')
        spikePath = os.path.join(l, 'spike_detection_refined_new.pkl')
        spikePath2 = os.path.join(l, 'spike_detection_refined_newr1.pkl')
        if os.path.exists(spikePath) or os.path.exists(spikePath2):
            if os.path.exists(spikePath):
                with open(spikePath, 'rb') as f:
                    spike_data = pickle.load(f)
                long_all_spikes = spike_data['vm_all_spikes']
                params_list = spike_data['detection_params']
                if len(params_list) > 0:
                    prev_params = params_list
                    init_CS = prev_params['pnorm_CS']
                    init_SS = prev_params['pnorm_SS']
                    init_thresh = prev_params['threshold']
            else:
                with open(spikePath2, 'rb') as f:
                    spike_data = pickle.load(f)
                long_all_spikes = spike_data['vm_all_spikes']
                params_list = data['detection_params']
                if len(params_list) > 0:
                    prev_params = params_list
                    init_CS = prev_params['pnorm_CS']
                    init_SS = prev_params['pnorm_SS']
                    init_thresh = prev_params['threshold']
        results = load_and_refine_pyr_spikes(
                    path=l,trace_vol=trace_vol,trace_cal=trace_cal,
                    frame_rate=500,
                    interactive=True,  # Set to True for interactive tuning
                    preW=75,
                    postW=100,spikeID=long_all_spikes,params_pr=prev_params

                )
        Anst_result.append(results)
        AnstSSVol.extend(results['vol_traces'])   
        AnstSSCal.extend(results['cal_traces'])
        AllCellAllData.append(results)

In [ ]:
#direct from basic
for i in range(82,len(pathPyr)):
    l = pathPyr[i]
    #print(l)
    TracePathCal = os.path.join(l,'calTraceDF.csv')
    TracePathVol = os.path.join(l,'volTraceDF.csv')
    TracePathCalM = os.path.join(l,'calMask.csv')
    TracePathVolM = os.path.join(l,'volMask.csv')
    parentP = os.path.dirname(l)
    MotPath = os.path.join(parentP,'Sync','MotorId.csv')
    trace_vol_full = np.array(pd.read_csv(TracePathVol)).flatten().astype(float)
    trace_cal_full = np.array(pd.read_csv(TracePathCal)).flatten().astype(float)
    vol_mask = np.array(pd.read_csv(TracePathVolM)).flatten().astype(bool)
    cal_mask = np.array(pd.read_csv(TracePathCalM)).flatten().astype(bool)
    # Apply masks
    trace_vol = trace_vol_full[vol_mask]
    trace_cal = trace_cal_full[cal_mask]
    motor = pd.read_csv(MotPath, header=None).iloc[:, 0]
    motor = motor[0:np.size(trace_vol,0)]
    VolAX = np.linspace(0, (len(trace_vol)/500), len(trace_vol)) 
    CalAX = np.linspace(0, (len(trace_cal)/30), len(trace_cal))
    path =os.path.join(l,r'SpikeIdx.csv')
    IspikeId = pd.read_csv(path)
    IspikeId = np.array(IspikeId)
    IspikeId = IspikeId.flatten()
    IspikeId = IspikeId.tolist()
    VolMask = vol_mask.astype(bool)

    # 2. Create a mapping from Old Index -> New Index
    # This creates an array where every index contains the count of "True" frames before it
    new_mapping = np.cumsum(vol_mask) - 1 

    # 3. Filter and Map
    # We keep the spike IF the mask was True at that spot...
    # ...AND we convert it to the new index using our mapping.
    spikeId = [new_mapping[int(i)] for i in IspikeId if int(i) < len(vol_mask) and vol_mask[int(i)]]
    long_all_spikes = sorted(list(set(spikeId)))
    long_all_spikes = sorted(spikeId)
    if bsPyr[i].lower()=='motor':
        TracePathCal = os.path.join(l,'changepoint.csv')
        changeP = pd.read_csv(TracePathCal)
        changeP = np.array(changeP)
        changeP = np.array(changeP).flatten().tolist()
        calMot,calRes, spikeMot,spikeRes,vmot,volR = Split_cal(changeP,trace_vol,trace_cal,VolAX,CalAX,motor,long_all_spikes)
        
        RestFRpath =os.path.join(l,r'FiringRateRest.csv')
        MotorFRpath =os.path.join(l,r'FiringRateMotor.csv')
        dfMot = pd.read_csv(MotorFRpath)
        MotorFR = dfMot['fr'].tolist()
        dfRest = pd.read_csv(RestFRpath)
        RestFR = dfRest['fr'].tolist()
        g = 0
        r = 0
        for k in range(len(MotorFR)):
            
            TracePathVol = os.path.join(l,f'volTraceMot{k}.csv')
            TracePathCal = os.path.join(l,f'calTraceMot{k}.csv')
            trace_vol_full = np.array(pd.read_csv(TracePathVol)).flatten().astype(float)
            trace_cal_full = np.array(pd.read_csv(TracePathCal)).flatten().astype(float)
            VolAX = np.linspace(0, (len(trace_vol_full)/500), len(trace_vol_full)) 
            CalAX = np.linspace(0, (len(trace_cal_full)/30), len(trace_cal_full))
            N = f'm{k}'
            r = r + 1
            curr_path_pkl = os.path.join(l, f'spike_detection_refined_new{N}.pkl')
            if os.path.exists(curr_path_pkl):
                with open(curr_path_pkl, 'rb') as f:
                    data = pickle.load(f)
                spike_indices = data['vm_all_spikes']
                
                params_list = data['detection_params']

                if len(params_list) > 0:
                    prev_params = params_list
                    init_CS = prev_params['pnorm_CS']
                    init_SS = prev_params['pnorm_SS']
                    init_thresh = prev_params['threshold']
                    INIT_CB_AMP = prev_params['cb_amp_threshold']
                    INIT_CB_DUR = prev_params['cb_duration_threshold']
                    init_thresh_SS = prev_params['simple_threshold_ss']
                    init_thresh_CS = prev_params['simple_threshold_cs']
                    init_thresh_ISI = prev_params['isi_threshold_ms']
            

                # Sort to ensure chronological order
                currSp = sorted(list(set(spike_indices)))
                currSp = sorted(spike_indices)
            else:
                currSp = spikeMot[k].tolist()
            if len(trace_vol_full) < 13 * 500:
                #print(f"SKIP {l} r{k}: VolTrace too short ({len(trace_vol_full)/500:.2f} s, {len(trace_vol_full)} frames)")
                continue
            plt.close('all')
            results = load_and_refine_pyr_spikes(
                    path=l,trace_vol=trace_vol_full,trace_cal=trace_cal_full,
                    frame_rate=500,
                    interactive=True,  # Set to True for interactive tuning
                    preW=75,
                    postW=100,spikeID=currSp,params_pr=prev_params,name=N

                )
            Motor_result.append(results)
            MotorSSVol.extend(results['vol_traces'])
            MotorSSCal.extend(results['cal_traces'])
            AllCellAllData.append(results)
        for j in range(len(RestFR)):
            TracePathVol = os.path.join(l,f'volTraceRest{j}.csv')
            TracePathCal = os.path.join(l,f'calTraceRest{j}.csv')
            trace_vol_full = np.array(pd.read_csv(TracePathVol)).flatten().astype(float)
            trace_cal_full = np.array(pd.read_csv(TracePathCal)).flatten().astype(float)
            VolAX = np.linspace(0, (len(trace_vol_full)/500), len(trace_vol_full)) 
            CalAX = np.linspace(0, (len(trace_cal_full)/30), len(trace_cal_full))
            N = f'r{j}'
            curr_path_pkl = os.path.join(l, f'spike_detection_refined_new{N}.pkl')
            if os.path.exists(curr_path_pkl):
                with open(curr_path_pkl, 'rb') as f:
                    data = pickle.load(f)
                spike_indices = data['vm_all_spikes']
                
                params_list = data['detection_params']
                if len(params_list) > 0:
                    prev_params = params_list
                    init_CS = prev_params['pnorm_CS']
                    init_SS = prev_params['pnorm_SS']
                    init_thresh = prev_params['threshold']

                # Sort to ensure chronological order
                currSp = sorted(list(set(spike_indices)))
                currSp = sorted(spike_indices)
            else:
                currSp = spikeRes[j].tolist()
            if len(trace_vol_full) < 10 * 500:
                print(f"SKIP {l} r{j}: VolTrace too short ({len(trace_vol_full)/500:.2f} s, {len(trace_vol_full)} frames)")
                continue
            plt.close('all')
            results = load_and_refine_pyr_spikes(
                    path=l,trace_vol=trace_vol_full,trace_cal=trace_cal_full,
                    frame_rate=500,
                    interactive=True,  # Set to True for interactive tuning
                    preW=75,
                    postW=100,spikeID=currSp,params_pr=prev_params,name=N

                )
            Rest_result.append(results)
            RestSSVol.extend(results['vol_traces'])
            RestSSCal.extend(results['cal_traces'])
            AllCellAllData.append(results)
    elif bsPyr[i].lower()=='awake':
        N=''
        plt.close('all')
        # spikePath = os.path.join(l, 'spike_detection_refined_new.pkl')
        # spikePath2 = os.path.join(l, 'spike_detection_refined_newr1.pkl')
        # if os.path.exists(spikePath) or os.path.exists(spikePath2):
        #     if os.path.exists(spikePath):
        #         with open(spikePath, 'rb') as f:
        #             spike_data = pickle.load(f)
        #         long_all_spikes = spike_data['vm_all_spikes']
        #         params_list = spike_data['detection_params']
        #         if len(params_list) > 0:
        #             prev_params = params_list
        #             init_CS = prev_params['pnorm_CS']
        #             init_SS = prev_params['pnorm_SS']
        #             init_thresh = prev_params['threshold']
        #     else:
        #         with open(spikePath2, 'rb') as f:
        #             spike_data = pickle.load(f)
        #         long_all_spikes = spike_data['vm_all_spikes']
        #         params_list = spike_data['detection_params']
        #         if len(params_list) > 0:
        #             prev_params = params_list
        #             init_CS = prev_params['pnorm_CS']
        #             init_SS = prev_params['pnorm_SS']
        #             init_thresh = prev_params['threshold']
        results = load_and_refine_pyr_spikes(
                    path=l,trace_vol=trace_vol,trace_cal=trace_cal,
                    frame_rate=500,
                    interactive=True,  # Set to True for interactive tuning
                    preW=75,
                    postW=100,spikeID=long_all_spikes,name=N

                )
        Awake_result.append(results)
        AwakeSSVol.extend(results['vol_traces'])   
        AwakeSSCal.extend(results['cal_traces'])
        AllCellAllData.append(results)
    elif bsPyr[i].lower()=='anst':
        N=''
        plt.close('all')
        
        results = load_and_refine_pyr_spikes(
                    path=l,trace_vol=trace_vol,trace_cal=trace_cal,
                    frame_rate=500,
                    interactive=True,  # Set to True for interactive tuning
                    preW=75,
                    postW=100,spikeID=long_all_spikes,name=N

                )
        Anst_result.append(results)
        AnstSSVol.extend(results['vol_traces'])   
        AnstSSCal.extend(results['cal_traces'])
        AllCellAllData.append(results)

In [6]:
#sst

DB = pd.read_csv(r'Z:\Adam-Lab-Shared\Data\Michal_Rubin\Dendrites\SST_F.csv')
values = DB['SNR'].tolist()
r = DB
awakePyr = r['Notes']
bsPyr = list(r['brainState'])
pathPyr = list(r['Link'])
AllCalSR = r['CALsr']
MotorSSVol = []
MotorSSCal = []
RestSSVol = []
RestSSCal = []
AwakeSSVol = []
AwakeSSCal = []
AnstSSVol = []
AnstSSCal = []
Rest_result = []
Motor_result = []
Anst_result = []
Awake_result = []
AllCellAllData= []

In [7]:
import numpy as np
import pickle
from pathlib import Path

# ---------------------------------------------------------
# Helpers
# ---------------------------------------------------------
def robust_z(x):
    x = np.asarray(x, float).ravel()
    med = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - med)) + 1e-12
    return (x - med) / (1.4826 * mad)

def compute_z_proxy_from_suite2p_ops(
    ops,
    use_vcorr=True,
    upsample_vcorr_if_needed=True,
    w_xy=0.2,
    w_corrxy=0.4,
    w_vcorr=0.4,
):
    xoff = np.asarray(ops.get("xoff", ops.get("xoff1", [])), float).ravel()
    yoff = np.asarray(ops.get("yoff", ops.get("yoff1", [])), float).ravel()
    if xoff.size == 0 or yoff.size == 0:
        raise KeyError("ops missing xoff/yoff.")

    n = min(xoff.size, yoff.size)
    motion_xy = np.sqrt(xoff[:n]**2 + yoff[:n]**2)
    xy_motion_z = robust_z(motion_xy)

    corrXY = ops.get("corrXY", ops.get("corrXY1", None))
    if corrXY is None:
        raise KeyError("ops missing corrXY.")
    corrXY_bad_z = robust_z(-np.asarray(corrXY, float).ravel()[:n])

    z_proxy = w_xy * xy_motion_z + w_corrxy * corrXY_bad_z
    weight_sum = w_xy + w_corrxy

    comps = {
        "xy_motion_z": xy_motion_z,
        "corrXY_bad_z": corrXY_bad_z,
    }

    if use_vcorr and "Vcorr" in ops:
        Vcorr = np.asarray(ops["Vcorr"], float).ravel()
        if Vcorr.size == n:
            vc = robust_z(-Vcorr)
        elif upsample_vcorr_if_needed and Vcorr.size > 2:
            x_old = np.linspace(0, n - 1, Vcorr.size)
            x_new = np.arange(n)
            vc = robust_z(-np.interp(x_new, x_old, Vcorr))
        else:
            vc = None

        if vc is not None:
            z_proxy += w_vcorr * vc
            weight_sum += w_vcorr
            comps["Vcorr_bad_z"] = vc

    z_proxy /= (weight_sum + 1e-12)
    return z_proxy, comps

def pick_bad_frames(z_proxy, method="mad", k=6.0, q=99.0):
    if method == "mad":
        med = np.nanmedian(z_proxy)
        mad = np.nanmedian(np.abs(z_proxy - med)) + 1e-12
        thr = med + k * mad
    else:
        thr = np.nanpercentile(z_proxy, q)

    bad = np.isfinite(z_proxy) & (z_proxy > thr)
    return bad, thr

def cal_bad_to_vol_bad_mask(bad_cal_mask, VolXaX, CalXax):
    bad_v = np.zeros(len(VolXaX), dtype=bool)
    dt_ca = np.median(np.diff(CalXax)) if len(CalXax) > 1 else 1.0

    bad_idx = np.where(bad_cal_mask)[0]
    for ci in bad_idx:
        t0 = CalXax[ci]
        t1 = CalXax[ci + 1] if ci < len(CalXax) - 1 else t0 + dt_ca
        bad_v |= (VolXaX >= t0) & (VolXaX < t1)

    return bad_v

# ---------------------------------------------------------
# Main Pipeline
# ---------------------------------------------------------
def remove_high_z_motion_both_modalities(
    cal_trace,
    vol_trace,
    *,
    ops,
    VolXaX,
    CalXax,
    mode="mask",
    thr_method="mad",
    thr_k=6.0,
    thr_q=99.0,
    pad_ca_frames=0,
    save_pickle_path=None,
):

    cal = np.asarray(cal_trace, float).ravel()
    vol = np.asarray(vol_trace, float).ravel()

    z_proxy, comps = compute_z_proxy_from_suite2p_ops(ops)

    n = min(len(cal), len(z_proxy))
    z_use = z_proxy[:n]

    bad_ca, thr = pick_bad_frames(
        z_use, method=thr_method, k=thr_k, q=thr_q
    )

    # Expand bad frames
    if pad_ca_frames > 0:
        expanded = bad_ca.copy()
        idx = np.where(bad_ca)[0]
        for i in idx:
            a = max(0, i - pad_ca_frames)
            b = min(n, i + pad_ca_frames + 1)
            expanded[a:b] = True
        bad_ca = expanded

    bad_v = cal_bad_to_vol_bad_mask(
        bad_ca,
        VolXaX=np.asarray(VolXaX),
        CalXax=np.asarray(CalXax),
    )

    if mode == "mask":
        cal_out = cal.copy()
        vol_out = vol.copy()
        cal_out[:n][bad_ca] = np.nan
        vol_out[bad_v] = np.nan
    else:
        keep_ca = np.ones_like(cal, dtype=bool)
        keep_ca[:n] = ~bad_ca
        cal_out = cal[keep_ca]
        vol_out = vol[~bad_v]

    out = {
        "cal_out": cal_out,
        "vol_out": vol_out,
        "bad_ca_mask": bad_ca,
        "bad_v_mask": bad_v,
        "bad_ca_idx": np.where(bad_ca)[0],
        "bad_v_idx": np.where(bad_v)[0],
        "z_proxy": z_use,
        "z_components": comps,
        "threshold": thr,
        "params": {
            "mode": mode,
            "thr_method": thr_method,
            "thr_k": thr_k,
            "thr_q": thr_q,
            "pad_ca_frames": pad_ca_frames,
        }
    }

    # -------------------------
    # Save pickle automatically
    # -------------------------
    if save_pickle_path is not None:
        save_path = Path(save_pickle_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        with open(save_path, "wb") as f:
            pickle.dump(out, f)
        print("Saved z-cleaning output to:", save_path)

    return out


In [14]:
import plotly.io as pio
pio.renderers.default = "browser"


In [15]:
import numpy as np
import plotly.graph_objects as go

def plot_z_proxy_threshold(out, *, CalXax=None, title="Z-shift proxy per frame (Suite2p)"):
    """
    out: dict returned by remove_high_z_motion_both_modalities(...)
    CalXax: optional time axis in seconds (len >= n_used). If None, uses frame index.
    """
    z = np.asarray(out["z_proxy"], float).ravel()
    thr = float(out["threshold"])
    bad = np.asarray(out["bad_ca_mask"], bool).ravel()

    # Align lengths (out["bad_ca_mask"] may be length n_used or full length)
    n = z.size
    bad = bad[:n] if bad.size >= n else np.pad(bad, (0, n - bad.size), constant_values=False)

    if CalXax is not None:
        x = np.asarray(CalXax, float).ravel()[:n]
        xlab = "Time (s)"
    else:
        x = np.arange(n)
        xlab = "Frame"

    fig = go.Figure()

    # z_proxy line
    fig.add_trace(go.Scatter(
        x=x, y=z, mode="lines",
        name="z_proxy (combined)"
    ))

    # threshold line
    fig.add_trace(go.Scatter(
        x=x, y=np.full(n, thr),
        mode="lines",
        name=f"threshold = {thr:.3g}",
        line=dict(dash="dash")
    ))

    # bad frames markers
    if bad.any():
        fig.add_trace(go.Scatter(
            x=x[bad], y=z[bad],
            mode="markers",
            name=f"bad frames ({bad.sum()})",
            marker=dict(size=6, symbol="x")
        ))

    # Optional component traces
    comps = out.get("z_components", {})
    for k, v in comps.items():
        v = np.asarray(v, float).ravel()
        if v.size == n:
            fig.add_trace(go.Scatter(
                x=x, y=v,
                mode="lines",
                name=k,
                opacity=0.45
            ))

    fig.update_layout(
        title=title,
        xaxis_title=xlab,
        yaxis_title="Proxy / robust z-score",
        template="plotly_white",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    )

    fig.show()
    return fig


In [17]:
#SST
MotorCalTran = []
MotorFrTran = []
RestCalTran = []
RestFrTran = []
AwakeCalTran = []
AwakeFrTran = []
AnstCalTran = []
AnstFrTran = []
MotorPears = []
MotorP_val = []
RestPears = []
RestP_val = []
AwakePears = []
AwakeP_val = []
AnstPears = []
AnstP_val = []
MotorCalDiff = []
MotorFRDiff = []
RestCalDiff = []
RestFRDiff = []
AwakeCalDiff = []
AwakeFRDiff = []
AnstCalDiff = []
AnstFRDiff = []
MotorOpt_r = []
MotorOpt_lag = []
RestOpt_r = []
RestOpt_lag = []
AwakeOpt_r = []
AwakeOpt_lag = []
AnstOpt_r = []
AnstOpt_lag = []
MotorWindow_r = []
MotorWindow_p = []
RestWindow_r = []
RestWindow_p = []
AwakeWindow_r = []
AwakeWindow_p = []
AnstWindow_r = []
AnstWindow_p = []
MotorFRMean = []
RestFRMean=[]
AwakeFRMean = []
AnstFRMean = []
MotorFRdiffMean = []
RestFRdiffMean=[]
AwakeFRdiffMean = []
AnstFRdiffMean = []
AllPath = []

calSR = 30
volSR =500
corr_summary_rows = []
for i,l in enumerate(pathPyr):
    calSR= AllCalSR[i]
    print(l)
    TracePathCal = os.path.join(l,'calTraceDF.csv')
    TracePathVol = os.path.join(l,'volTraceDF.csv')
    TracePathCalM = os.path.join(l,'calMask.csv')
    TracePathVolM = os.path.join(l,'volMask.csv')
    parentP = os.path.dirname(l)
    MotPath = os.path.join(parentP,'Sync','MotorId.csv')
    
    
    VolTrace = pd.read_csv(TracePathVol)
    VolTrace = np.array(VolTrace)
    VolTrace = VolTrace.flatten()
    Trace = VolTrace
    motor = pd.read_csv(MotPath, header=None).iloc[:, 0]
    motor = motor[0:np.size(Trace,0)]
    CalTrace = pd.read_csv(TracePathCal)
    CalTrace = np.array(CalTrace)
    CalTrace = CalTrace.flatten()
    VolMask = pd.read_csv(TracePathVolM)
    VolMask = np.array(VolMask)
    VolMask = VolMask.flatten()
    #Trace = VolMask 
    CalMask = pd.read_csv(TracePathCalM)
    CalMask = np.array(CalMask)
    CalMask = CalMask.flatten()
    TraceV = Trace[VolMask]
    TraceC = CalTrace[CalMask]
    motor= motor[VolMask]
    VolAX = np.linspace(0, (len(TraceV)/500), len(TraceV)) 
    CalAX = np.linspace(0, (len(TraceC)/calSR), len(TraceC))
    fpath = os.path.join(l,r'SpikeIdxFinal.csv')
    if os.path.exists(fpath):
        pathSpike = fpath
        spikeId = pd.read_csv(pathSpike)
        spikeId = np.array(spikeId)
        spikeId = spikeId.flatten()
        spikeId = spikeId.tolist()
    else:
        pathSpike = os.path.join(l,r'SpikeIdx.csv')
        print('LOL')
        IspikeId = pd.read_csv(pathSpike)
        IspikeId = np.array(IspikeId)
        IspikeId = IspikeId.flatten()
        IspikeId = IspikeId.tolist()
        VolMask = VolMask.astype(bool)

        # 2. Create a mapping from Old Index -> New Index
        # This creates an array where every index contains the count of "True" frames before it
        new_mapping = np.cumsum(VolMask) - 1 

        # 3. Filter and Map
        # We keep the spike IF the mask was True at that spot...
        # ...AND we convert it to the new index using our mapping.
        spikeId = [new_mapping[int(i)] for i in IspikeId if int(i) < len(VolMask) and VolMask[int(i)]]
    if bsPyr[i].lower()=='motor':
        N =''
        AllPath.append(l)
        spk = sst_spike_correction_gui(TraceV, fs=500, save_dir=l,name = N,chunk_s=30)
    #     TracePathCal = os.path.join(l,'changepoint.csv')
    #     changeP = pd.read_csv(TracePathCal)
    #     changeP = np.array(changeP)
    #     changeP = np.array(changeP).flatten().tolist()
    #     #calMot,calRes = Split_cal(changeP,TraceV,TraceC,VolAX,CalAX,motor)
    #     MotorSinglePikeIDX = []
    #     RestSinglePikeIDX = []
    #     MotorSinglePikeCalIDX = []
    #     RestSinglePikeCalIDX = []
    #     MotorBurstSpikeIDX = []
    #     RestBurstSpikeIDX = []
    #     MotorComplexSpikeIDX = []
    #     RestComplexSpikeIDX = []
    #     RestFRpath =os.path.join(l,r'FiringRateRest.csv')
    #     MotorFRpath =os.path.join(l,r'FiringRateMotor.csv')
        
    #     dfMot = pd.read_csv(MotorFRpath)
    #     MotorFR = dfMot['fr'].tolist()
    #     dfRest = pd.read_csv(RestFRpath)
    #     RestFR = dfRest['fr'].tolist()
    #     for k in range(len(MotorFR)):
            
    #         N = f'm{k}'
    #         TracePathVol = os.path.join(l,f'volTraceMot{k}.csv')
    #         TracePathCal = os.path.join(l,f'calTraceMot{k}.csv')
    #         VolTrace = pd.read_csv(TracePathVol)
    #         VolTrace = np.array(VolTrace)
    #         VolTrace = VolTrace.flatten()
    #         VolAX = np.linspace(0, (len(VolTrace)/500), len(VolTrace)) 
    #         CalTrace = pd.read_csv(TracePathCal)
    #         CalTrace = np.array(CalTrace)
    #         CalTrace = CalTrace.flatten()
    #         CalAX = np.linspace(0, (len(CalTrace)/calSR), len(CalTrace))
    #         CorrSPPath =  os.path.join(l,f'spikeTraceMot{k}.csv')
    #         spikeId = pd.read_csv(CorrSPPath)
    #         spikeId = np.array(spikeId)
    #         spikeId = spikeId.flatten()
    #         spikeId = spikeId.tolist()
    #         txtPss = os.path.join(l,'SinglS_par.txt')
    #         spk = sst_spike_correction_gui(VolTrace, fs=500, save_dir=l,name = N)

    #     for j in range(len(RestFR)):
            
    #         N = f'r{j}'
    #         TracePathVol = os.path.join(l,f'volTraceRest{j}.csv')
    #         TracePathCal = os.path.join(l,f'calTraceRest{j}.csv')
    #         VolTrace = pd.read_csv(TracePathVol)
    #         VolTrace = np.array(VolTrace)
    #         VolTrace = VolTrace.flatten()
    #         VolAX = np.linspace(0, (len(VolTrace)/500), len(VolTrace)) 
    #         CalTrace = pd.read_csv(TracePathCal)
    #         CalTrace = np.array(CalTrace)
    #         CalTrace = CalTrace.flatten()
    #         CalAX = np.linspace(0, (len(CalTrace)/calSR), len(CalTrace))
    #         CorrSPPath =  os.path.join(l,f'spikeTraceMot{j}.csv')
    #         spikeId = pd.read_csv(CorrSPPath)
    #         spikeId = np.array(spikeId)
    #         spikeId = spikeId.flatten()
    #         spikeId = spikeId.tolist()
    #         txtPss = os.path.join(l,'SinglS_par.txt')
    #         spk = sst_spike_correction_gui(VolTrace, fs=500, save_dir=l,name = N)
    # elif bsPyr[i].lower()=='awake':
    #     N = ''
    #     AllPath.append(l)
    #     fpath = os.path.join(l,r'SpikeIdxFinal.csv')
    #     if os.path.exists(fpath):
    #         pathSpike = fpath
    #         spikeId = pd.read_csv(pathSpike)
    #         spikeId = np.array(spikeId)
    #         spikeId = spikeId.flatten()
    #         spikeId = spikeId.tolist()
    #     else:
    #         pathSpike = os.path.join(l,r'SpikeIdx.csv')
    #         print('LOL')
    #         IspikeId = pd.read_csv(pathSpike)
    #         IspikeId = np.array(IspikeId)
    #         IspikeId = IspikeId.flatten()
    #         IspikeId = IspikeId.tolist()
    #         VolMask = VolMask.astype(bool)

    #         # 2. Create a mapping from Old Index -> New Index
    #         # This creates an array where every index contains the count of "True" frames before it
    #         new_mapping = np.cumsum(VolMask) - 1 

    #         # 3. Filter and Map
    #         # We keep the spike IF the mask was True at that spot...
    #         # ...AND we convert it to the new index using our mapping.
    #         spikeId = [new_mapping[int(i)] for i in IspikeId if int(i) < len(VolMask) and VolMask[int(i)]]
    #     txtPss = os.path.join(l,'SinglS_par.txt')
    #     spk = sst_spike_correction_gui(TraceV, fs=500, save_dir=l,name = N,chunk_s=30)
    # elif bsPyr[i].lower()=='anst':
    #     N = ''
    #     AllPath.append(l)
    #     fpath = os.path.join(l,r'SpikeIdxFinal.csv')
    #     if os.path.exists(fpath):
    #         pathSpike = fpath
    #         spikeId = pd.read_csv(pathSpike)
    #         spikeId = np.array(spikeId)
    #         spikeId = spikeId.flatten()
    #         spikeId = spikeId.tolist()
    #     else:
    #         pathSpike = os.path.join(l,r'SpikeIdx.csv')
    #         print('LOL')
    #         IspikeId = pd.read_csv(pathSpike)
    #         IspikeId = np.array(IspikeId)
    #         IspikeId = IspikeId.flatten()
    #         IspikeId = IspikeId.tolist()
    #         VolMask = VolMask.astype(bool)

    #         # 2. Create a mapping from Old Index -> New Index
    #         # This creates an array where every index contains the count of "True" frames before it
    #         new_mapping = np.cumsum(VolMask) - 1 

    #         # 3. Filter and Map
    #         # We keep the spike IF the mask was True at that spot...
    #         # ...AND we convert it to the new index using our mapping.
    #         spikeId = [new_mapping[int(i)] for i in IspikeId if int(i) < len(VolMask) and VolMask[int(i)]]
    #     spk = sst_spike_correction_gui(TraceV, fs=500, save_dir=l,name = N,chunk_s=50)


Z:\Adam-Lab-Shared\Data\Michal_Rubin\srugc17\Xb\17-06-2025\fov1\cell1
LOL


KeyboardInterrupt: 